# 04 Export Viz JSON -- Vision Zero LA

**Workspace:** CTS-Geo Analytics GBD  
**Lakehouse:** VisionZero_Lakehouse  

**Purpose:** Read Gold Delta tables and export them as compact JSON files
for consumption by the CityPulse Traffic Flow web application.

**Outputs (written to Lakehouse Files):**
- `vz-heatmap.json` -- H3 hexagonal crash heatmap
- `vz-corridors.json` -- Road segments colored by crash density
- `vz-hotspots.json` -- Getis-Ord Gi* hot spot bins
- `vz-emerging.json` -- Space-time emerging hot spot trends
- `vz-crashes.json` -- Individual crash points
- `vz-stats.json` -- Summary statistics + monthly time series

In [ ]:
%pip install h3

In [ ]:
import os
import json
import h3
from pyspark.sql import functions as F

LAKEHOUSE = "VisionZero_Lakehouse"
OUTPUT_BASE = "/lakehouse/default/Files/citypulse_output"

os.makedirs(OUTPUT_BASE, exist_ok=True)
print(f"Output directory: {OUTPUT_BASE}")

In [ ]:
# ── vz-heatmap.json ──────────────────────────────────────────────────────────
#
# Format: array of {h3, boundary, count, fatal, ped, bike, severity_score, res, lat, lng}

heatmap_df = spark.read.format("delta").table(f"gold_h3_heatmap")
heatmap_rows = heatmap_df.collect()

heatmap_data = []
for row in heatmap_rows:
    # Use h3 library to get boundary coords if not stored, otherwise parse from column
    try:
        boundary = json.loads(row.boundary)
    except Exception:
        try:
            raw_boundary = h3.cell_to_boundary(row.h3_index)
            boundary = [[round(lng, 6), round(lat, 6)] for lat, lng in raw_boundary]
            boundary.append(boundary[0])
        except Exception:
            continue

    heatmap_data.append({
        "h3": row.h3_index,
        "boundary": boundary,
        "count": int(row.crash_count),
        "fatal": int(row.fatal_count),
        "ped": int(row.pedestrian_count),
        "bike": int(row.bicycle_count),
        "severity_score": int(row.severity_score) if row.severity_score else 0,
        "res": int(row.resolution),
        "lat": round(float(row.center_lat), 5),
        "lng": round(float(row.center_lng), 5),
    })

out_path = f"{OUTPUT_BASE}/vz-heatmap.json"
with open(out_path, "w") as f:
    json.dump(heatmap_data, f)

print(f"Wrote vz-heatmap.json: {len(heatmap_data):,} hexagons")

In [ ]:
# ── vz-corridors.json ────────────────────────────────────────────────────────
#
# Format: array of {path, color, crashes, fatal, pedestrian, bicycle, road_name}

try:
    corridors_df = spark.read.format("delta").table(f"gold_danger_corridors")
    corridor_rows = corridors_df.collect()

    corridor_data = []
    for row in corridor_rows:
        try:
            path = json.loads(row.path_json)
            color = json.loads(row.color_json)
        except Exception:
            continue

        corridor_data.append({
            "path": path,
            "color": color,
            "crashes": int(row.crash_count),
            "fatal": int(row.fatal),
            "pedestrian": int(row.pedestrian),
            "bicycle": int(row.bicycle),
            "road_name": row.road_name or "",
        })

    out_path = f"{OUTPUT_BASE}/vz-corridors.json"
    with open(out_path, "w") as f:
        json.dump(corridor_data, f)

    print(f"Wrote vz-corridors.json: {len(corridor_data):,} road segments")

except Exception as e:
    print(f"Skipped vz-corridors.json: {e}")

In [ ]:
# ── vz-hotspots.json ─────────────────────────────────────────────────────────
#
# Format: array of {bin_id, boundary, crash_count, z_score, p_value, classification}

try:
    hotspots_df = spark.read.format("delta").table(f"gold_hotspots")
    hotspot_rows = hotspots_df.collect()

    hotspot_data = []
    for row in hotspot_rows:
        entry = {
            "bin_id": str(getattr(row, "bin_id", "")),
            "crash_count": int(getattr(row, "count", 0)),
            "z_score": round(float(getattr(row, "z_score", 0)), 4),
            "p_value": round(float(getattr(row, "p_value", 1)), 6),
            "classification": str(getattr(row, "classification", "")),
        }

        # Extract boundary geometry if available
        if hasattr(row, "geometry") and row.geometry is not None:
            try:
                entry["boundary"] = json.loads(str(row.geometry))
            except Exception:
                entry["boundary"] = None
        elif hasattr(row, "boundary") and row.boundary is not None:
            try:
                entry["boundary"] = json.loads(row.boundary)
            except Exception:
                entry["boundary"] = None
        else:
            entry["boundary"] = None

        hotspot_data.append(entry)

    out_path = f"{OUTPUT_BASE}/vz-hotspots.json"
    with open(out_path, "w") as f:
        json.dump(hotspot_data, f)

    print(f"Wrote vz-hotspots.json: {len(hotspot_data):,} bins")

except Exception as e:
    print(f"Skipped vz-hotspots.json: {e}")

In [ ]:
# ── vz-emerging.json ─────────────────────────────────────────────────────────
#
# Format: array of {bin_id, boundary, trend, quarters_hot, latest_z}

try:
    emerging_df = spark.read.format("delta").table(f"gold_emerging_hotspots")
    emerging_rows = emerging_df.collect()

    emerging_data = []
    for row in emerging_rows:
        entry = {
            "bin_id": str(getattr(row, "bin_id", "")),
            "trend": str(getattr(row, "trend", "")),
            "quarters_hot": int(getattr(row, "quarters_hot", 0)) if hasattr(row, "quarters_hot") else 0,
            "latest_z": round(float(getattr(row, "latest_z", 0)), 4) if hasattr(row, "latest_z") else 0.0,
        }

        # Extract boundary geometry
        if hasattr(row, "geometry") and row.geometry is not None:
            try:
                entry["boundary"] = json.loads(str(row.geometry))
            except Exception:
                entry["boundary"] = None
        elif hasattr(row, "boundary") and row.boundary is not None:
            try:
                entry["boundary"] = json.loads(row.boundary)
            except Exception:
                entry["boundary"] = None
        else:
            entry["boundary"] = None

        emerging_data.append(entry)

    out_path = f"{OUTPUT_BASE}/vz-emerging.json"
    with open(out_path, "w") as f:
        json.dump(emerging_data, f)

    print(f"Wrote vz-emerging.json: {len(emerging_data):,} bins")

except Exception as e:
    print(f"Skipped vz-emerging.json: {e}")

In [ ]:
# ── vz-crashes.json ──────────────────────────────────────────────────────────
#
# Format: array of {lat, lng, date, hour, sev, desc, loc, cross, ym}

silver_df = spark.read.format("delta").table(f"silver_crashes")

crash_rows = (
    silver_df
    .select(
        "latitude", "longitude", "crash_date", "crash_hour",
        "severity", "crm_cd_desc", "location", "cross_street", "year_month"
    )
    .collect()
)

crash_data = []
for row in crash_rows:
    crash_data.append({
        "lat": round(float(row.latitude), 5),
        "lng": round(float(row.longitude), 5),
        "date": row.crash_date.isoformat() if row.crash_date else None,
        "hour": int(row.crash_hour) if row.crash_hour is not None else None,
        "sev": row.severity,
        "desc": row.crm_cd_desc,
        "loc": row.location,
        "cross": row.cross_street,
        "ym": row.year_month,
    })

out_path = f"{OUTPUT_BASE}/vz-crashes.json"
with open(out_path, "w") as f:
    json.dump(crash_data, f)

print(f"Wrote vz-crashes.json: {len(crash_data):,} crash points")

In [ ]:
# ── vz-stats.json ────────────────────────────────────────────────────────────
#
# Combines gold_stats and gold_timeseries into one JSON object.

# Read stats
stats_df = spark.read.format("delta").table(f"gold_stats")
stats_row = stats_df.collect()[0]

stats_obj = {
    "total_crashes": int(stats_row.total_crashes),
    "fatal": int(stats_row.fatal),
    "pedestrian": int(stats_row.pedestrian),
    "bicycle": int(stats_row.bicycle),
    "severe": int(stats_row.severe),
    "other": int(stats_row.other),
    "date_from": stats_row.date_from,
    "date_to": stats_row.date_to,
    "school_zone_crashes": int(stats_row.school_zone_crashes),
    "school_zone_fatal": int(stats_row.school_zone_fatal),
    "top_intersections": json.loads(stats_row.top_intersections_json),
    "by_severity": json.loads(stats_row.by_severity_json),
}

# Read timeseries
ts_df = spark.read.format("delta").table(f"gold_timeseries")
ts_rows = ts_df.orderBy("year_month").collect()

stats_obj["timeseries"] = [
    {
        "month": row.year_month,
        "total": int(row.total),
        "fatal": int(row.fatal),
        "pedestrian": int(row.pedestrian),
        "bicycle": int(row.bicycle),
        "severe": int(row.severe),
        "other": int(row.other),
    }
    for row in ts_rows
]

out_path = f"{OUTPUT_BASE}/vz-stats.json"
with open(out_path, "w") as f:
    json.dump(stats_obj, f, indent=2)

print(f"Wrote vz-stats.json")
print(f"  Date range: {stats_obj['date_from']} to {stats_obj['date_to']}")
print(f"  Total crashes: {stats_obj['total_crashes']:,}")
print(f"  Time series months: {len(stats_obj['timeseries'])}")

In [ ]:
# ── Verify all output files ──────────────────────────────────────────────────

expected_files = [
    "vz-heatmap.json",
    "vz-corridors.json",
    "vz-hotspots.json",
    "vz-emerging.json",
    "vz-crashes.json",
    "vz-stats.json",
]

print("=" * 60)
print("Output File Verification")
print("=" * 60)

all_ok = True
for fname in expected_files:
    fpath = f"{OUTPUT_BASE}/{fname}"
    if os.path.exists(fpath):
        size_bytes = os.path.getsize(fpath)
        if size_bytes > 1024 * 1024:
            size_str = f"{size_bytes / (1024 * 1024):.1f} MB"
        else:
            size_str = f"{size_bytes / 1024:.1f} KB"
        print(f"  OK  {fname:25s}  {size_str:>10s}")
    else:
        print(f"  MISSING  {fname}")
        all_ok = False

print("=" * 60)
if all_ok:
    print("All files verified.")
else:
    print("WARNING: Some files are missing. Check the Gold notebooks ran successfully.")

In [ ]:
# ── Completion ────────────────────────────────────────────────────────────────

print("=" * 60)
print("Vision Zero LA -- Export Complete")
print("=" * 60)
print(f"\nAll JSON files written to: {OUTPUT_BASE}/")
print("\nTo use in the CityPulse web app:")
print("  1. Download files from Lakehouse Files > citypulse_output/")
print("  2. Copy to the web app data/ directory")
print("  3. Or serve via OneLake API / Fabric SQL endpoint")
print("\nFile mapping for the web app:")
print("  vz-heatmap.json   -> H3 hexagonal crash overlay")
print("  vz-corridors.json -> Animated road trails with danger colors")
print("  vz-hotspots.json  -> Getis-Ord Gi* cluster visualization")
print("  vz-emerging.json  -> Emerging trend overlay")
print("  vz-crashes.json   -> Individual crash point markers")
print("  vz-stats.json     -> Stats panel + time series playback")
print("\nSchedule this pipeline via Fabric Pipeline for weekly refresh.")